# Token Trajectory: How Representations Evolve Layer by Layer

**Based on:** Elhage et al., *"A Mathematical Framework for Transformer Circuits"* (Anthropic, 2021)

## The Residual Stream Concept

In a transformer, each token's representation flows through the network via the **residual stream**. At each layer, two sub-modules — **self-attention** and the **MLP (feed-forward network)** — read from the residual stream, compute updates, and *add* their contributions back. The output of layer $l$ is:

$$x^{(l)} = x^{(l-1)} + \text{Attn}^{(l)}(x^{(l-1)}) + \text{MLP}^{(l)}(x^{(l-1)} + \text{Attn}^{(l)}(x^{(l-1)}))$$

This additive structure means every layer's contribution is a **delta** on top of the previous representation. By hooking into each layer, we can track exactly how each token's representation evolves — where it changes most, when its "meaning" crystallizes, and whether attention or the MLP drives the change.

In this notebook, we instrument all 32 layers of **Llama 3.1 8B** with PyTorch hooks to capture the full hidden state trajectory for each token in a prompt.

## 1. Imports and Model Loading

In [ ]:
%matplotlib inline

import os
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
from transformers import AutoTokenizer, AutoModelForCausalLM
from pathlib import Path

matplotlib.rcParams.update({"font.size": 12})

MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"
CACHE_DIR = os.path.expanduser("~/models")
OUTPUT_DIR = os.path.expanduser("~/hook_outputs/charts")

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, cache_dir=CACHE_DIR)

print("Loading model...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    cache_dir=CACHE_DIR,
    torch_dtype=torch.float16,
    device_map="auto",
)
model.eval()

num_layers = model.config.num_hidden_layers
print("Model loaded: {} layers, hidden size {}".format(num_layers, model.config.hidden_size))

## 2. Register Hooks on All Layers

We hook into three locations per layer:
- **Layer output** (full residual stream after the layer) — captured via hooks on each `model.model.layers[i]`
- **Self-attention output** — captured via hooks on each `model.model.layers[i].self_attn`
- **MLP output** — captured via hooks on each `model.model.layers[i].mlp`

We also capture the **embedding output** (before any transformer layer) to establish the baseline representation.

In [ ]:
PROMPT = "The capital of France is Paris"

inputs = tokenizer(PROMPT, return_tensors="pt").to(model.device)
input_ids = inputs["input_ids"]
tokens = [tokenizer.decode(t) for t in input_ids[0]]
num_tokens = len(tokens)

print("Prompt: '{}'".format(PROMPT))
print("Tokens ({}): {}".format(num_tokens, tokens))

# Storage for hook outputs
layer_outputs = {}      # layer index -> hidden state tensor [1, seq_len, hidden_dim]
attn_outputs = {}       # layer index -> attention sub-layer output
mlp_outputs = {}        # layer index -> mlp sub-layer output
embedding_output = {}   # single entry for the embedding

# --- Hook functions ---
def make_layer_hook(layer_idx):
    def hook_fn(module, input, output):
        # For decoder layers, output is a tuple; hidden states are the first element
        hidden = output[0] if isinstance(output, tuple) else output
        layer_outputs[layer_idx] = hidden.detach().cpu().float()
    return hook_fn

def make_attn_hook(layer_idx):
    def hook_fn(module, input, output):
        # Self-attention returns a tuple: (attn_output, attn_weights, past_key_value)
        hidden = output[0] if isinstance(output, tuple) else output
        attn_outputs[layer_idx] = hidden.detach().cpu().float()
    return hook_fn

def make_mlp_hook(layer_idx):
    def hook_fn(module, input, output):
        hidden = output.detach().cpu().float() if isinstance(output, torch.Tensor) else output[0].detach().cpu().float()
        mlp_outputs[layer_idx] = hidden
    return hook_fn

def embedding_hook(module, input, output):
    embedding_output["embed"] = output.detach().cpu().float()

# --- Register hooks ---
hooks = []

# Hook on the embedding layer
hooks.append(model.model.embed_tokens.register_forward_hook(embedding_hook))

# Hook on each transformer layer, its self_attn, and its mlp
for i in range(num_layers):
    hooks.append(model.model.layers[i].register_forward_hook(make_layer_hook(i)))
    hooks.append(model.model.layers[i].self_attn.register_forward_hook(make_attn_hook(i)))
    hooks.append(model.model.layers[i].mlp.register_forward_hook(make_mlp_hook(i)))

print("Registered {} hooks ({} layers x 3 + 1 embedding)".format(len(hooks), num_layers))

# --- Forward pass ---
with torch.no_grad():
    _ = model(**inputs)

print("Forward pass complete. Captured {} layer outputs, {} attn outputs, {} mlp outputs.".format(
    len(layer_outputs), len(attn_outputs), len(mlp_outputs)
))

# Remove hooks
for h in hooks:
    h.remove()
print("All hooks removed.")

In [ ]:
# Build the full trajectory: embedding + 32 layer outputs = 33 states
# trajectory shape: (33, num_tokens, hidden_dim)

trajectory = []

# Layer 0 = embedding output
trajectory.append(embedding_output["embed"][0])  # [seq_len, hidden_dim]

# Layers 1..32 = output of each transformer layer
for i in range(num_layers):
    trajectory.append(layer_outputs[i][0])  # [seq_len, hidden_dim]

trajectory = torch.stack(trajectory, dim=0)  # [33, seq_len, hidden_dim]
print("Trajectory shape: {} (stages x tokens x hidden_dim)".format(trajectory.shape))

# Labels for the stages
stage_labels = ["embed"] + ["layer_{}".format(i) for i in range(num_layers)]

## 3. Visualization 1 — Cosine Similarity Between Consecutive Layers (Heatmap)

For each token position, we compute the cosine similarity between its representation at layer $l$ and layer $l+1$. A value close to 1.0 means the representation barely changed; a lower value indicates a significant transformation.

This heatmap reveals **where in the network** each token's representation undergoes the most change.

In [ ]:
# Compute cosine similarity between consecutive layers for each token
# Result shape: (32, num_tokens) — 32 transitions between 33 stages

num_stages = trajectory.shape[0]
cosine_consecutive = np.zeros((num_stages - 1, num_tokens))

for s in range(num_stages - 1):
    for t in range(num_tokens):
        vec_a = trajectory[s, t]
        vec_b = trajectory[s + 1, t]
        sim = F.cosine_similarity(vec_a.unsqueeze(0), vec_b.unsqueeze(0)).item()
        cosine_consecutive[s, t] = sim

# Plot heatmap
fig, ax = plt.subplots(figsize=(max(8, num_tokens * 1.2), 12))
im = ax.imshow(cosine_consecutive, aspect="auto", cmap="RdYlGn", vmin=0.85, vmax=1.0)

ax.set_xlabel("Token Position")
ax.set_ylabel("Layer Transition")
ax.set_title("Cosine Similarity Between Consecutive Layers\n(per token position)")

# Token labels on x-axis
ax.set_xticks(range(num_tokens))
ax.set_xticklabels(tokens, rotation=45, ha="right", fontsize=10)

# Layer transition labels on y-axis
transition_labels = ["{} -> {}".format(stage_labels[i], stage_labels[i + 1]) for i in range(num_stages - 1)]
ax.set_yticks(range(0, num_stages - 1, 2))
ax.set_yticklabels([transition_labels[i] for i in range(0, num_stages - 1, 2)], fontsize=8)

cbar = plt.colorbar(im, ax=ax, shrink=0.8)
cbar.set_label("Cosine Similarity")

plt.tight_layout()
save_path = os.path.join(OUTPUT_DIR, "chart_17_cosine_trajectory.png")
plt.savefig(save_path, dpi=150, bbox_inches="tight")
plt.show()
print("Saved: {}".format(save_path))

## 4. Visualization 2 — Token Resolution Curve

For a specific token (e.g., "France"), we track how its representation at each layer compares to its **final-layer** representation. A cosine similarity approaching 1.0 means the model has "resolved" this token's contextual meaning.

This shows the layer at which the token's representation converges to its final form.

In [ ]:
# Find the token index for "France"
target_token = "France"
target_idx = None
for i, tok in enumerate(tokens):
    if target_token.lower() in tok.lower():
        target_idx = i
        break

if target_idx is None:
    print("WARNING: '{}' not found in tokens. Using token index 3.".format(target_token))
    target_idx = 3
else:
    print("Found '{}' at token index {} (decoded: '{}')".format(target_token, target_idx, tokens[target_idx]))

# Cosine similarity of each layer's representation vs the final layer
final_repr = trajectory[-1, target_idx]  # final layer representation
sim_to_final = []

for s in range(num_stages):
    current_repr = trajectory[s, target_idx]
    sim = F.cosine_similarity(current_repr.unsqueeze(0), final_repr.unsqueeze(0)).item()
    sim_to_final.append(sim)

# Plot
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(range(num_stages), sim_to_final, marker="o", linewidth=2, markersize=5, color="tab:blue")
ax.fill_between(range(num_stages), sim_to_final, alpha=0.15, color="tab:blue")

ax.set_xlabel("Layer (0 = embedding)")
ax.set_ylabel("Cosine Similarity to Final Layer")
ax.set_title("Token Resolution: '{}' — Similarity to Final Representation Across Layers".format(tokens[target_idx].strip()))
ax.set_xticks(range(0, num_stages, 2))
ax.set_xticklabels([stage_labels[i] for i in range(0, num_stages, 2)], rotation=45, ha="right", fontsize=8)
ax.set_ylim(0, 1.05)
ax.axhline(y=0.95, color="red", linestyle="--", alpha=0.5, label="95% threshold")
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
save_path = os.path.join(OUTPUT_DIR, "chart_18_token_resolution.png")
plt.savefig(save_path, dpi=150, bbox_inches="tight")
plt.show()
print("Saved: {}".format(save_path))

## 5. Visualization 3 — Attention vs MLP Contribution Norms (Stacked Bar Chart)

At each layer, we measure the L2 norm of the self-attention output and the MLP output. These represent the magnitude of each sub-module's contribution to the residual stream update.

A stacked bar chart decomposes what drives changes at each layer.

In [ ]:
# Compute average L2 norm of attention and MLP outputs across all token positions
attn_norms = []
mlp_norms = []

for i in range(num_layers):
    # Average L2 norm across token positions
    attn_norm = torch.norm(attn_outputs[i][0], dim=-1).mean().item()
    mlp_norm = torch.norm(mlp_outputs[i][0], dim=-1).mean().item()
    attn_norms.append(attn_norm)
    mlp_norms.append(mlp_norm)

attn_norms = np.array(attn_norms)
mlp_norms = np.array(mlp_norms)

# Stacked bar chart
fig, ax = plt.subplots(figsize=(14, 6))
x = np.arange(num_layers)

bars_attn = ax.bar(x, attn_norms, label="Self-Attention", color="tab:blue", alpha=0.8)
bars_mlp = ax.bar(x, mlp_norms, bottom=attn_norms, label="MLP", color="tab:orange", alpha=0.8)

ax.set_xlabel("Layer")
ax.set_ylabel("L2 Norm of Contribution")
ax.set_title("Attention vs MLP Contribution Norm per Layer\n(averaged across token positions)")
ax.set_xticks(range(0, num_layers, 2))
ax.set_xticklabels([str(i) for i in range(0, num_layers, 2)])
ax.legend()
ax.grid(True, alpha=0.3, axis="y")

plt.tight_layout()
save_path = os.path.join(OUTPUT_DIR, "chart_19_attn_vs_mlp_contribution.png")
plt.savefig(save_path, dpi=150, bbox_inches="tight")
plt.show()
print("Saved: {}".format(save_path))

## 6. Visualization 4 — L2 Norm of Residual Stream Deltas (Line Plot)

Instead of looking at raw sub-module output norms, here we compute the **actual change** to the residual stream at each layer — the L2 norm of `layer_output[l] - layer_output[l-1]`. We then decompose this by comparing it to the attention and MLP norms from above.

This tracks how much the residual stream *actually moves* at each layer.

In [ ]:
# Compute per-layer L2 norm of the difference added by attention vs MLP
# We use the sub-module outputs directly as they represent the additive contributions

attn_delta_norms = []
mlp_delta_norms = []

for i in range(num_layers):
    # Attention contribution: L2 norm averaged across tokens
    attn_delta = torch.norm(attn_outputs[i][0], dim=-1).mean().item()
    mlp_delta = torch.norm(mlp_outputs[i][0], dim=-1).mean().item()
    attn_delta_norms.append(attn_delta)
    mlp_delta_norms.append(mlp_delta)

# Line plot
fig, ax = plt.subplots(figsize=(14, 6))
layers_x = range(num_layers)

ax.plot(layers_x, attn_delta_norms, marker="s", linewidth=2, markersize=5,
        color="tab:blue", label="Attention Contribution (L2 norm)")
ax.plot(layers_x, mlp_delta_norms, marker="^", linewidth=2, markersize=5,
        color="tab:orange", label="MLP Contribution (L2 norm)")

# Also plot the total residual stream delta
total_deltas = []
for i in range(num_layers):
    # Delta between consecutive residual stream states
    if i == 0:
        delta = trajectory[i + 1] - trajectory[0]  # layer_0 output - embedding
    else:
        delta = trajectory[i + 1] - trajectory[i]   # layer_i output - layer_(i-1) output
    total_delta = torch.norm(delta, dim=-1).mean().item()
    total_deltas.append(total_delta)

ax.plot(layers_x, total_deltas, marker="o", linewidth=2, markersize=5,
        color="tab:green", label="Total Residual Delta (L2 norm)", linestyle="--")

ax.set_xlabel("Layer")
ax.set_ylabel("L2 Norm")
ax.set_title("L2 Norm of Attention vs MLP Contributions per Layer\n(averaged across token positions)")
ax.set_xticks(range(0, num_layers, 2))
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
save_path = os.path.join(OUTPUT_DIR, "chart_20_delta_norms.png")
plt.savefig(save_path, dpi=150, bbox_inches="tight")
plt.show()
print("Saved: {}".format(save_path))

## Summary

This notebook traced **token representations through all 32 layers** of Llama 3.1 8B using PyTorch forward hooks. Key findings:

1. **Consecutive-layer cosine similarity** (Chart 17): Reveals that early layers (especially the transition from embedding to layer 0) produce the largest representational shifts. Later layers tend to make more incremental adjustments, though specific layers may show notable drops for particular tokens.

2. **Token resolution curve** (Chart 18): Tracks when a token's meaning "crystallizes" by measuring similarity to its final-layer representation. Content-bearing tokens like "France" typically converge earlier in the network than function words, consistent with the residual stream gradually accumulating contextual information.

3. **Attention vs MLP stacked norms** (Chart 19): Decomposes each layer's contribution into its two sub-modules. The MLP contribution often dominates in middle-to-late layers, while attention tends to be more prominent in early layers where positional and syntactic patterns are established.

4. **Delta norm line plot** (Chart 20): Shows the magnitude of change at each layer, with separate traces for attention, MLP, and total residual delta. The total delta is generally bounded by the sum of attention and MLP norms, confirming the additive structure of the residual stream described in Elhage et al. (2021).

**Reference:** Elhage, N. et al. (2021). *A Mathematical Framework for Transformer Circuits.* Anthropic.